In [12]:
import pandas as pd
import seaborn as sns

pd.set_option("display.max_columns", 50)
sns.set_theme(style="whitegrid")

In [3]:
df = pd.read_csv("../data/raw/online_retail_II.csv", encoding="ISO-8859-1")
df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [5]:
df = df.rename(columns={"Customer ID": "CustomerID"})
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])
df["Invoice"] = df["Invoice"].astype(str)
print(df.columns.tolist())

['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'CustomerID', 'Country']


In [6]:
print(df.shape)
print(df["InvoiceDate"].min(), df["InvoiceDate"].max())
df.info()
print(df.isna().sum())
print("дубликатов:", df.duplicated().sum())

(1067371, 8)
2009-12-01 07:45:00 2011-12-09 12:50:00
<class 'pandas.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 8 columns):
 #   Column       Non-Null Count    Dtype         
---  ------       --------------    -----         
 0   Invoice      1067371 non-null  str           
 1   StockCode    1067371 non-null  str           
 2   Description  1062989 non-null  str           
 3   Quantity     1067371 non-null  int64         
 4   InvoiceDate  1067371 non-null  datetime64[us]
 5   Price        1067371 non-null  float64       
 6   CustomerID   824364 non-null   float64       
 7   Country      1067371 non-null  str           
dtypes: datetime64[us](1), float64(2), int64(1), str(4)
memory usage: 117.3 MB
Invoice             0
StockCode           0
Description      4382
Quantity            0
InvoiceDate         0
Price               0
CustomerID     243007
Country             0
dtype: int64
дубликатов: 34335


In [7]:
n_raw = len(df)

cancelled = df["Invoice"].str.startswith("C").sum()
neg_qty = (df["Quantity"] <= 0).sum()
bad_price = (df["Price"] <= 0).sum()

service_pattern = r"^(POST|D|M|BANK CHARGES|PADS|DOT|CRUK|AMAZONFEE|B|S|ADJUST.*|TEST.*)$"
service = df["StockCode"].astype(str).str.match(service_pattern, case=False).sum()

print("всего строк:", n_raw)
print("отмены (Invoice на C):", cancelled)
print("Quantity <= 0:", neg_qty)
print("Price <= 0:", bad_price)
print("служебные коды:", service)

print(df["StockCode"][df["StockCode"].astype(str).str.match(service_pattern, case=False)].value_counts())
print(df[["Quantity", "Price"]].describe())

всего строк: 1067371
отмены (Invoice на C): 19494
Quantity <= 0: 22950
Price <= 0: 6207
служебные коды: 5548
StockCode
POST            2122
DOT             1446
M               1421
D                177
S                104
BANK CHARGES     102
ADJUST            67
AMAZONFEE         43
PADS              19
CRUK              16
TEST001           15
B                  6
m                  5
ADJUST2            3
TEST002            2
Name: count, dtype: int64
           Quantity         Price
count  1.067371e+06  1.067371e+06
mean   9.938898e+00  4.649388e+00
std    1.727058e+02  1.235531e+02
min   -8.099500e+04 -5.359436e+04
25%    1.000000e+00  1.250000e+00
50%    3.000000e+00  2.100000e+00
75%    1.000000e+01  4.150000e+00
max    8.099500e+04  3.897000e+04


In [8]:
log = []
def step(name, data):
    log.append((name, len(data)))
    return data

d = step("исходные данные", df.copy())

returns = d[d["Invoice"].str.startswith("C")].copy()

d = step("без CustomerID", d.dropna(subset=["CustomerID"]))
d = step("без отмен (Invoice на C)", d[~d["Invoice"].str.startswith("C")])
d = step("Quantity > 0 и Price > 0", d[(d["Quantity"] > 0) & (d["Price"] > 0)])
d = step("без служебных кодов", d[~d["StockCode"].astype(str).str.match(service_pattern, case=False)])
d = step("без полных дубликатов", d.drop_duplicates())

clean = d.copy()
clean["CustomerID"] = clean["CustomerID"].astype(int)
clean["Revenue"] = clean["Quantity"] * clean["Price"]

log_df = pd.DataFrame(log, columns=["шаг", "строк осталось"])
log_df["удалено на шаге"] = (-log_df["строк осталось"].diff()).fillna(0).astype(int)
log_df

,шаг,строк осталось,удалено на шаге
0,исходные данные,1067371,0
1,без CustomerID,824364,243007
2,без отмен (Invoice на C),805620,18744
3,Quantity > 0 и Price > 0,805549,71
4,без служебных кодов,802887,2662
5,без полных дубликатов,776827,26060


In [9]:
print(f"было {n_raw}, стало {len(clean)}, удалено {n_raw - len(clean)} ({(n_raw - len(clean)) / n_raw:.1%})")
print("клиентов:", clean["CustomerID"].nunique())
print("заказов:", clean["Invoice"].nunique())
print("период:", clean["InvoiceDate"].min().date(), "-", clean["InvoiceDate"].max().date())
print("выручка всего:", round(clean["Revenue"].sum()))
print("страны:", clean["Country"].nunique(), "| доля UK:", round((clean["Country"] == "United Kingdom").mean(), 3))

clean.to_parquet("../data/processed/clean_transactions.parquet", index=False)
returns.to_parquet("../data/processed/returns.parquet", index=False)

было 1067371, стало 776827, удалено 290544 (27.2%)
клиентов: 5853
заказов: 36600
период: 2009-12-01 - 2011-12-09
выручка всего: 17081129
страны: 41 | доля UK: 0.901
